# 图

输出到 `figures/`。每格自己读输入，可以单独运行：热图和方差曲线读 `results/` 下的 CSV；
山脊图和箱线图读特征、重新做 PCA（几秒），不依赖 `analysis.ipynb` 的结果。

## PC × 标签热图

In [ ]:
import pandas as pd

from fmspace.paths import FIGURE_DIR, RESULT_DIR
from fmspace.plots import heatmap

assoc = pd.read_csv(RESULT_DIR / "pc_association.csv")
for name in assoc["model"].unique():
    fig = heatmap(assoc, name, ["eye_dr_level", "artifact", "clarity", "eye"])
    fig.savefig(FIGURE_DIR / f"heatmap_{name}.pdf", bbox_inches="tight")

## 累计解释方差

In [ ]:
import pandas as pd

from fmspace.paths import FIGURE_DIR, RESULT_DIR
from fmspace.plots import variance_curve

variance = pd.read_csv(RESULT_DIR / "explained_variance.csv")
fig = variance_curve(variance, variance["model"].unique(), n_pcs=50, mark=20)
fig.savefig(FIGURE_DIR / "variance_cumulative.pdf", bbox_inches="tight")

## 各等级在 PC1 上的分布（山脊图）

PC1 得分做 z-score，方向调成与 DR 正相关；4 个模型共用横轴范围。每个等级各自归一化，虚线为中位数。

In [ ]:
from fmspace.analysis import l2, pca, zscore
from fmspace.data import load_metadata
from fmspace.extract import load_features
from fmspace.models import MODELS
from fmspace.paths import DATA_DIR, FEATURE_DIR, FIGURE_DIR
from fmspace.plots import grade_ridgeline

metadata = load_metadata(DATA_DIR)
y = metadata["eye_dr_level"].to_numpy()
pc1 = {}
for name in MODELS:
    X = l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
    pc1[name] = zscore(pca(X).transform(X)[:, 0], y)
xlim = (min(s.min() for s in pc1.values()), max(s.max() for s in pc1.values()))
for name in MODELS:
    fig = grade_ridgeline(pc1[name], y, name, xlim)
    fig.savefig(FIGURE_DIR / f"ridgeline_pc1_{name}.pdf", bbox_inches="tight")

## 各等级在 PC1 上的箱线图（4 个模型并排）

PC1 得分做 z-score，方向调成与 DR 正相关。

In [ ]:
from fmspace.analysis import l2, pca, zscore
from fmspace.data import load_metadata
from fmspace.extract import load_features
from fmspace.models import MODELS
from fmspace.paths import DATA_DIR, FEATURE_DIR, FIGURE_DIR
from fmspace.plots import grade_boxplot

metadata = load_metadata(DATA_DIR)
y = metadata["eye_dr_level"].to_numpy()
pc1 = {}
for name in MODELS:
    X = l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
    pc1[name] = zscore(pca(X).transform(X)[:, 0], y)
fig = grade_boxplot(pc1, y, list(MODELS))
fig.savefig(FIGURE_DIR / "boxplot_pc1.pdf", bbox_inches="tight")